# 4. Attribution fidelity and calibrated uncertainty

This is the notebook the project exists for. Two questions:

1. **Is the explanation correct?** Scored against exact ground truth, and against two controls -- a uniform-random attribution, and the same method applied to an *untrained* model (the parameter-randomisation sanity check of Adebayo et al., 2018).
2. **Does the model know when it is wrong?** Interval coverage, sharpness, and a risk-coverage curve for abstention.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd, torch
torch.set_num_threads(2)
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
tables = pathlib.Path('../results/tables')
fid = pd.read_csv(tables / 'attribution_fidelity.csv') if (tables / 'attribution_fidelity.csv').exists() else None
cols = ['model', 'attribution', 'joint_precision', 'joint_recall',
        'joint_iou', 'joint_rank_corr', 'joint_top1_hit',
        'frame_localisation_error', 'n_joint_iou']
fid[[c for c in cols if c in fid.columns]].round(4) if fid is not None else 'run `make compare` first'

**How to read this.** The rows to compare against are `random` and `integrated_gradients_untrained`. An attribution method that does not beat both is not explaining the model -- it is reflecting the input statistics. Where that happens it is reported as a negative result.

In [ ]:
from saqa.viz import plot_attribution_fidelity
p = plot_attribution_fidelity('nb_attribution_fidelity.png')
print(p); plt.show()

## A single sequence, explained

In [ ]:
from saqa.config import load_config
from saqa.pipelines import make_splits, run_single, compute_attribution
from saqa.viz import plot_attribution_map
cfg = load_config('../configs/base.yaml', ['data.num_sequences=300', 'optim.epochs=4', 'run.out_dir=../results/runs_nb'])
sp = make_splits(cfg)
res = run_single(cfg, name='nb_attr', splits=sp, save=False, verbose=False)
k = int(np.argmax(sp.test.joint_truth.sum(1) > 0))
attr = compute_attribution(res.model, sp.test.coords[k:k+1], 'integrated_gradients', cfg)[0]
plot_attribution_map(attr, sp.test.joint_truth[k], sp.test.frame_truth[k],
                     name='nb_attribution_map.png'); plt.show()
print(sp.test.samples[k].action, [d.kind for d in sp.test.samples[k].degradations])

## Monotonicity: does adding a defect ever *raise* the score?

The ordinal head guarantees rank consistency (its survival function cannot cross itself), but it does **not** guarantee monotonicity in the input. That is an empirical question and it is measured, not claimed.

In [ ]:
from saqa.pipelines import monotonicity_check, ordinal_consistency
rep, ladders, preds = monotonicity_check(res.model, cfg, num_ladders=24)
print({k: round(v, 4) for k, v in rep.items() if not k.startswith('n__')})
print(ordinal_consistency(res.model, sp.test))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for lad, pr in list(zip(ladders, preds))[:8]:
    ax.plot(lad['severity'], pr, 'o-', alpha=0.7, label=lad['kind'])
ax.set_xlabel('defect severity'); ax.set_ylabel('predicted quality')
ax.set_title('every line should slope down'); ax.grid(alpha=0.3)
ax.legend(fontsize=7); plt.show()

## Intervals: coverage next to sharpness

Coverage alone cannot distinguish a useful interval from a constant-width one, so the width conditional on the model being wrong is reported alongside. A ratio above 1 means the interval widens where it should.

In [ ]:
p = tables / 'uncertainty.csv'
pd.read_csv(p).round(4) if p.exists() else 'run `make compare` first'

In [ ]:
from saqa.viz import plot_risk_coverage
p = plot_risk_coverage('../results/runs/saqa_stgcn/per_item.csv',
                       name='nb_risk_coverage.png')
print(p); plt.show() if p else print('run `make compare` first')